In [4]:
import os
import re
import shutil
import zipfile
import numpy as np
import pandas as pd

# === SETTINGS ===
input_folder = r'C:\Users\dugue\PycharmProjects\design_of_experiment_for_nuclear_fuels\RawFuels'
output_folder = os.path.join(input_folder, 'processed_pinns')

# Set True so stale long-format CSVs from the broken preprocessor are deleted.
# This only deletes the processed_pinns folder, not the raw Excel files.
CLEAR_OUTPUT_FOLDER = True

if CLEAR_OUTPUT_FOLDER and os.path.isdir(output_folder):
    print(f"Deleting old processed output folder: {output_folder}")
    shutil.rmtree(output_folder)

os.makedirs(output_folder, exist_ok=True)

# Sheets to process:
# key = expected sheet name in workbook
# value = subfolder/output label
SHEETS_TO_PROCESS = {
    'Total HGR': 'Total_HGR',
    'Burnup': 'BURNUP',
}

# If True, files without FUEL_DATA_MAP entries are skipped.
# If False, they are processed with blank/NaN TD_Density and N_U-235.
SKIP_IF_LOOKUP_MISSING = False

# This preprocessor now writes the model-loader-friendly WIDE format:
#     9 static columns + one column per timestep.
# Do not add Fuel, TD_Percent, N_U-238 here unless you also update load_data.py.
MODEL_STATIC_COLUMNS = [
    'Enrichment',
    'TD_Density',
    'Irradiation_Vehicle',
    'R',
    'A',
    'S',
    'N_U-235',
    'Radial_R',
    'Axial_Z',
]

# === COMMON TIME STAMPS ===
TIMESTEPS = [
    0, 1, 3, 5, 10, 15, 20, 22, 24, 25, 25.001, 26, 28, 30, 35, 40, 45, 47, 49, 50,
    50.001, 51, 53, 55, 60, 65, 70, 72, 74, 75, 75.001, 76, 78, 80, 85, 90, 95, 97,
    99, 100, 100.001, 101, 103, 105, 110, 115, 120, 122, 124, 125, 125.001, 126,
    128, 130, 135, 140, 145, 147, 149, 150
]

# === LOOKUP DICTIONARIES ===
FUEL_DATA_MAP = {
    ('UC', 0.35, 80, 'RB'): (10.904, 9.3103e19, 2.6174e22),
    ('UC', 0.35, 90, 'RB'): (12.267, 1.0474e20, 2.9445e22),
    ('UC', 0.35, 100, 'RB'): (13.63, 1.1638e20, 3.2717e22),
    ('UC', 5.0, 90, 'RB'): (13.63, 1.1638e20, 2.8071e22),

    ('UN', 0.35, 80, 'RB'): (11.472, 9.7175e19, 2.7318e22),
    ('UN', 0.35, 90, 'RB'): (12.906, 1.0978e20, 3.0862e22),
    ('UN', 0.35, 100, 'RB'): (14.34, 1.2147e20, 3.4148e22),
    ('UN', 5.0, 90, 'RB'): (12.906, 1.5683e21, 2.9422e22),

    ('UO2', 0.35, 80, 'RB'): (8.768, 6.9319e19, 1.9487e22),
    ('UO2', 0.35, 90, 'RB'): (9.864, 7.7984e19, 2.1923e22),
    ('UO2', 0.35, 100, 'RB'): (10.96, 8.6649e19, 2.4359e22),

    ('UO2', 0.711, 80, 'RB'): (8.768, 1.4082e20, 1.9417e22),
    ('UO2', 0.711, 90, 'RB'): (9.864, 1.5842e20, 2.1844e22),
    ('UO2', 0.711, 100, 'RB'): (10.96, 1.7602e20, 2.4271e22),

    ('UO2', 5.0, 90, 'RB'): (9.864, 1.1141e21, 2.0900e22),

    ('UO2', 0.35, 90, 'VXF'): (9.864, 7.7984e19, 2.1923e22),
    ('UO2', 0.711, 90, 'VXF'): (9.864, 1.5842e20, 2.1844e22),
    ('UO2', 5.0, 90, 'VXF'): (9.864, 1.1141e21, 2.0900e22),
}

AXIAL_MAP = {
    'VXF': {
        (3, 6): 254.611, (3, 5): 230.111, (3, 4): 205.611, (3, 3): 181.111, (3, 2): 156.611, (3, 1): 132.111,
        (2, 6): 54.65,   (2, 5): 30.15,   (2, 4): 5.65,    (2, 3): -18.85,  (2, 2): -43.35,  (2, 1): -67.85,
        (1, 6): -145.3115, (1, 5): -169.8115, (1, 4): -194.3115, (1, 3): -218.8115, (1, 2): -243.3115, (1, 1): -267.8115
    },
    'RB': {
        (3, 6): 276.611, (3, 5): 252.111, (3, 4): 227.611, (3, 3): 203.111, (3, 2): 178.611, (3, 1): 154.111,
        (2, 6): 76.65,   (2, 5): 52.15,   (2, 4): 27.65,   (2, 3): 3.15,    (2, 2): -21.35,  (2, 1): -45.85,
        (1, 6): -123.3115, (1, 5): -147.8115, (1, 4): -172.3115, (1, 3): -196.8115, (1, 2): -221.3115, (1, 1): -245.8115
    }
}

RADIAL_MAP = {
    'VXF': {1: 401.32, 2: 387.505, 3: 387.505},
    'RB':  {1: 260.3889, 2: 260.3889, 3: 277.9464, 4: 288.7, 5: 277.9464}
}

VALID_FUELS = {'UO2', 'UC', 'UN'}
VALID_IVS = {'RB', 'VXF'}


def strip_summary_suffix(base_name):
    return re.sub(r'_summary$', '', base_name, flags=re.IGNORECASE)


def parse_enrichment_token(token):
    # Handles .35, 0.35, 0.711, 5, 5.0
    return round(float(str(token).strip()), 6)


def parse_td_token(token):
    # Handles 80, 90, 100, PTD090, PDT090, TD090
    token = str(token).upper().strip()
    match = re.search(r'(\d+)', token)
    if not match:
        raise ValueError(f"Could not parse theoretical density from token: {token}")
    return int(match.group(1).lstrip('0') or '0')


def parse_filename(base_name):
    """
    Supported filename formats:

    1. RB-.35-100-UO2_summary.xlsx       -> IV-Enrichment-TD-Fuel
    2. RB-0.35-90-UC_summary.xlsx       -> IV-Enrichment-TD-Fuel
    3. UO2-0.711-90-RB_summary.xlsx     -> Fuel-Enrichment-TD-IV
    4. 0.35-PTD090-VXF_summary.xlsx     -> Enrichment-PTD/PDT-IV, assumes UO2
    """
    name = strip_summary_suffix(base_name)
    parts = [p.strip().upper() for p in name.split('-')]

    if len(parts) == 4 and parts[0] in VALID_IVS and parts[3] in VALID_FUELS:
        iv = parts[0]
        enrichment = parse_enrichment_token(parts[1])
        td_pct = parse_td_token(parts[2])
        fuel = parts[3]
        return fuel, enrichment, td_pct, iv

    if len(parts) == 4 and parts[0] in VALID_FUELS and parts[3] in VALID_IVS:
        fuel = parts[0]
        enrichment = parse_enrichment_token(parts[1])
        td_pct = parse_td_token(parts[2])
        iv = parts[3]
        return fuel, enrichment, td_pct, iv

    if len(parts) == 3 and parts[2] in VALID_IVS and re.search(r'P[TD][TD]\d+', parts[1]):
        fuel = 'UO2'
        enrichment = parse_enrichment_token(parts[0])
        td_pct = parse_td_token(parts[1])
        iv = parts[2]
        return fuel, enrichment, td_pct, iv

    raise ValueError(f"Unsupported filename format: {base_name} -> parts={parts}")


def lookup_fuel_data(fuel, enrichment, td_pct, iv):
    # Tolerant lookup so 5 and 5.0 behave the same.
    for (f, e, td, vehicle), values in FUEL_DATA_MAP.items():
        if (
            f == fuel
            and abs(float(e) - float(enrichment)) < 1e-9
            and int(td) == int(td_pct)
            and vehicle == iv
        ):
            return values
    return None, None, None


def get_file_signature(path, num_bytes=8):
    try:
        with open(path, 'rb') as f:
            return f.read(num_bytes)
    except Exception:
        return b''


def detect_excel_engine(path):
    ext = os.path.splitext(path)[1].lower()
    signature = get_file_signature(path)

    is_zip = zipfile.is_zipfile(path)
    is_ole = signature.startswith(b'\xD0\xCF\x11\xE0')

    if is_zip:
        return "openpyxl", "Detected ZIP-based Excel workbook."
    if is_ole:
        return "xlrd", "Detected old binary Excel workbook. Requires xlrd installed."
    if ext == ".xlsx":
        return "openpyxl", (
            "File has .xlsx extension but does not look like a ZIP-based .xlsx file. "
            "Trying openpyxl anyway; if this fails, re-save it from Excel as Excel Workbook (*.xlsx)."
        )
    if ext == ".xls":
        return "xlrd", "File has .xls extension. Trying xlrd. If this fails, install xlrd with: pip install xlrd"
    return None, "Unknown Excel-like file type."


def open_excel_file(path, file_name):
    engine, diagnostic = detect_excel_engine(path)
    if engine is None:
        raise RuntimeError(f"No Excel engine detected. {diagnostic}")

    try:
        xls = pd.ExcelFile(path, engine=engine)
        return xls, engine
    except ImportError as e:
        if engine == "xlrd":
            raise RuntimeError(
                f"{file_name} appears to need xlrd, but xlrd is not installed. "
                f"Install it with: pip install xlrd. Original error: {e}"
            )
        raise
    except Exception as first_error:
        message = (
            f"Could not open {file_name} with engine='{engine}'.\n"
            f"Diagnostic: {diagnostic}\n"
            f"Original error: {first_error}\n"
        )
        if not zipfile.is_zipfile(path) and os.path.splitext(path)[1].lower() == ".xlsx":
            message += (
                "This is probably not a real .xlsx file even though the extension says .xlsx. "
                "Open it in Excel and use File -> Save As -> Excel Workbook (*.xlsx), then rerun.\n"
            )
        raise RuntimeError(message)


def clean_ras_string(val):
    s = str(val).strip()
    if s.endswith('.0'):
        s = s[:-2]
    s = ''.join(c for c in s if c.isdigit())
    return s.zfill(3)


def clean_columns(columns):
    # Do not use df.columns.str.strip(); it can damage numeric timestep headers.
    return [c.strip() if isinstance(c, str) else c for c in columns]


def find_column(df, allowed_substrings):
    for c in df.columns:
        c_lower = str(c).strip().lower()
        if any(sub in c_lower for sub in allowed_substrings):
            return c
    return None


def as_float_or_none(x):
    if isinstance(x, str) and x.strip().lower().startswith('unnamed'):
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return float(str(x).strip())
    except (ValueError, TypeError):
        return None


def get_time_columns(df):
    """
    Finds columns whose headers are numeric and match the expected timestep list.
    This is safer than slicing after metadata columns.
    """
    time_cols = []
    time_values = []

    for c in df.columns:
        value = as_float_or_none(c)
        if value is None:
            continue
        if any(abs(value - t) <= 1e-9 for t in TIMESTEPS):
            time_cols.append(c)
            time_values.append(value)

    # Keep the order from TIMESTEPS, not whatever Excel happened to provide.
    paired = sorted(zip(time_cols, time_values), key=lambda p: TIMESTEPS.index(next(t for t in TIMESTEPS if abs(p[1] - t) <= 1e-9)))
    time_cols = [p[0] for p in paired]
    time_values = [p[1] for p in paired]

    return time_cols, time_values


def safe_name(value):
    return str(value).strip().replace(" ", "_").replace("/", "_").replace("\\", "_")


def find_actual_sheet_name(xls, desired_sheet_name):
    desired = desired_sheet_name.strip().lower()
    for s in xls.sheet_names:
        if s.strip().lower() == desired:
            return s
    return None


def build_model_static_metadata(comp_df, ras_col, fuel, enrichment, td_pct, iv, true_density, nu_235_value):
    """
    Build exactly the 9 static columns expected by load_data.py.
    This deliberately excludes Fuel, TD_Percent, and N_U-238 so the output is
    directly model-compatible.
    """
    ras_str = comp_df[ras_col].apply(clean_ras_string)

    meta_df = pd.DataFrame(index=comp_df.index)
    meta_df['Enrichment'] = enrichment
    meta_df['TD_Density'] = true_density
    meta_df['Irradiation_Vehicle'] = iv

    meta_df['R'] = ras_str.str[0].astype(int)
    meta_df['A'] = ras_str.str[1].astype(int)
    meta_df['S'] = ras_str.str[2].astype(int)

    meta_df['N_U-235'] = nu_235_value
    meta_df['Radial_R'] = meta_df['R'].map(RADIAL_MAP.get(iv, {}))
    meta_df['Axial_Z'] = meta_df.apply(
        lambda row: AXIAL_MAP.get(iv, {}).get((row['A'], row['S']), np.nan),
        axis=1,
    )

    return meta_df[MODEL_STATIC_COLUMNS]


def save_wide_csv(base_name, material_type, output_value_col, meta_df, time_data):
    """
    Save one row per RAS/material row and one column per timestep.

    Output schema:
        Enrichment, TD_Density, Irradiation_Vehicle, R, A, S,
        N_U-235, Radial_R, Axial_Z, 0, 1, 3, 5, ...
    """
    result = pd.concat([meta_df.reset_index(drop=True), time_data.reset_index(drop=True)], axis=1)

    static_columns = MODEL_STATIC_COLUMNS
    time_columns = [c for c in result.columns if c not in static_columns]
    time_columns = sorted(time_columns, key=lambda x: float(x))

    result = result[static_columns + time_columns].reset_index(drop=True)

    safe_mat = safe_name(material_type)
    safe_sheet = safe_name(output_value_col)

    sheet_folder = os.path.join(output_folder, safe_mat, safe_sheet)
    os.makedirs(sheet_folder, exist_ok=True)

    output_file = f"{base_name}_{safe_mat}_{safe_sheet}.csv"
    output_path = os.path.join(sheet_folder, output_file)

    result.to_csv(output_path, index=False)
    return output_path, len(result), len(time_columns)


# ============================================================
# Main loop
# ============================================================

processed_files = 0
processed_outputs = 0
skipped_files = 0
missing_lookup_keys = []
bad_excel_files = []

for file in os.listdir(input_folder):
    if file.startswith('~$'):
        continue

    if not file.lower().endswith(('.xlsx', '.xls')):
        continue

    file_path = os.path.join(input_folder, file)
    base_name = os.path.splitext(file)[0]

    print(f"\n📂 Processing File: {file}")

    try:
        fuel, enrichment, td_pct, iv = parse_filename(base_name)
    except Exception as e:
        print(f"⚠️ Skipping {file}: {e}")
        skipped_files += 1
        continue

    print(f"Parsed: fuel={fuel}, enrichment={enrichment}, td_pct={td_pct}, iv={iv}")

    true_density, nu_235_value, nu_238_value = lookup_fuel_data(
        fuel=fuel,
        enrichment=enrichment,
        td_pct=td_pct,
        iv=iv,
    )

    if true_density is None:
        key = (fuel, enrichment, td_pct, iv)
        missing_lookup_keys.append(key)
        print(f"⚠️ No lookup data found for key: {key}")

        if SKIP_IF_LOOKUP_MISSING:
            print("   Skipping because SKIP_IF_LOOKUP_MISSING=True")
            skipped_files += 1
            continue

    try:
        xls, engine = open_excel_file(file_path, file)
        print(f"Opened with engine='{engine}'")
    except Exception as e:
        print(f"❌ Could not open {file}: {e}")
        bad_excel_files.append(file)
        skipped_files += 1
        continue

    processed_files += 1

    for target_sheet, output_value_col in SHEETS_TO_PROCESS.items():
        sheet_name_actual = find_actual_sheet_name(xls, target_sheet)

        if sheet_name_actual is None:
            print(f"⚠️ Sheet not found in {file}: {target_sheet}")
            continue

        print(f"  Processing sheet: {sheet_name_actual}")

        try:
            df = pd.read_excel(file_path, sheet_name=sheet_name_actual, engine=engine)
        except Exception as e:
            print(f"❌ Could not read {file} [{sheet_name_actual}] with engine='{engine}': {e}")
            continue

        df.columns = clean_columns(df.columns)

        ras_col = find_column(df, ['ras'])
        mat_col = find_column(df, ['material', 'component'])

        if ras_col is None or mat_col is None:
            print(f"❌ Could not find RAS or Material/Component column in {file} [{sheet_name_actual}]. Skipping sheet.")
            print(f"   Columns found: {list(df.columns)}")
            continue

        time_cols, time_values = get_time_columns(df)

        if len(time_cols) == 0:
            print(f"❌ Could not find timestep columns in {file} [{sheet_name_actual}]. Skipping sheet.")
            print(f"   Columns found: {list(df.columns)}")
            continue

        if len(time_cols) != len(TIMESTEPS):
            print(f"⚠️ Expected {len(TIMESTEPS)} timesteps but found {len(time_cols)} in {file} [{sheet_name_actual}].")
            print("   Saving the timesteps actually present in the workbook.")

        for material_type in df[mat_col].dropna().unique():
            comp_df = df[df[mat_col] == material_type].copy().reset_index(drop=True)

            if comp_df.empty:
                continue

            meta_df = build_model_static_metadata(
                comp_df=comp_df,
                ras_col=ras_col,
                fuel=fuel,
                enrichment=enrichment,
                td_pct=td_pct,
                iv=iv,
                true_density=true_density,
                nu_235_value=nu_235_value,
            )

            # Wide time-series data: columns are the actual timestep headers.
            time_data = comp_df[time_cols].copy()
            time_data = time_data.apply(pd.to_numeric, errors='coerce')
            time_data.columns = time_values

            output_path, n_rows, n_steps = save_wide_csv(
                base_name=base_name,
                material_type=material_type,
                output_value_col=output_value_col,
                meta_df=meta_df,
                time_data=time_data,
            )

            processed_outputs += 1
            print(f"    ✅ Saved WIDE: {output_path} rows={n_rows} timesteps={n_steps}")


# ============================================================
# Final report
# ============================================================

print("\n✅ Done.")
print(f"Processed workbook files: {processed_files}")
print(f"Created CSV outputs: {processed_outputs}")
print(f"Skipped files: {skipped_files}")

if missing_lookup_keys:
    print("\n⚠️ Missing lookup keys encountered:")
    for key in sorted(set(missing_lookup_keys)):
        print(f"  {key}")

if bad_excel_files:
    print("\n❌ Files that could not be opened as Excel workbooks:")
    for file in bad_excel_files:
        print(f"  {file}")

    print("\nFor these files, try opening them manually in Excel and using:")
    print("  File -> Save As -> Excel Workbook (*.xlsx)")
    print("Then rerun this script.")



📂 Processing File: 0.35-PTD090-VXF_summary.xlsx
Parsed: fuel=UO2, enrichment=0.35, td_pct=90, iv=VXF
Opened with engine='openpyxl'
  Processing sheet: Total HGR
    ✅ Saved WIDE: C:\Users\dugue\PycharmProjects\design_of_experiment_for_nuclear_fuels\RawFuels\processed_pinns\SS_housing\Total_HGR\0.35-PTD090-VXF_summary_SS_housing_Total_HGR.csv rows=9 timesteps=60
    ✅ Saved WIDE: C:\Users\dugue\PycharmProjects\design_of_experiment_for_nuclear_fuels\RawFuels\processed_pinns\Top_Spring\Total_HGR\0.35-PTD090-VXF_summary_Top_Spring_Total_HGR.csv rows=9 timesteps=60
    ✅ Saved WIDE: C:\Users\dugue\PycharmProjects\design_of_experiment_for_nuclear_fuels\RawFuels\processed_pinns\Bot_Spring\Total_HGR\0.35-PTD090-VXF_summary_Bot_Spring_Total_HGR.csv rows=9 timesteps=60
    ✅ Saved WIDE: C:\Users\dugue\PycharmProjects\design_of_experiment_for_nuclear_fuels\RawFuels\processed_pinns\Fuel\Total_HGR\0.35-PTD090-VXF_summary_Fuel_Total_HGR.csv rows=54 timesteps=60
    ✅ Saved WIDE: C:\Users\dugue\Pych